# Full score range

Анализ распределений и воронки отборов для выбранного чекпоинта.


In [ ]:
CHECKPOINT = ("261021_2009_da_nu_classifier_exp_full_E1_weights_"
              "z128_k10_dlr1e-2_lam0p003_delay2_s42_raking@best_da_model")


In [ ]:
import os, sys
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")
ROOT = "/home/albert/Baikal2025"
sys.path.insert(0, ROOT)
sys.path.insert(0, ROOT + "/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")
os.chdir("/home/albert/Baikal2025/inference/nu_classifier/analysis/HIGH_SCORE_EXCESS")  # относительные пути как в исходных ноутбуках

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from inference import reader
from excess.full import (load_events, attach_truth, domain_frames,
                          quality, by_class, ORDER)

pd.set_option("display.width", 200)

events, covered = load_events(CHECKPOINT)
events = attach_truth(events)
_frames = domain_frames(events)
exp, muatm, nuatm, nue2 = (_frames["exp"], _frames["muatm"],
                           _frames["nuatm"], _frames["nue2"])


In [ ]:
passes = quality(events)
funnel = pd.DataFrame({
    "сырых": by_class(covered, events.data_class),
    "фит_сошёлся": by_class(covered & (events.status == "ok"), events.data_class),
    "каты_реконструкции": by_class(covered & passes, events.data_class),
    "проскорено": by_class(covered & passes & events.score.notna(), events.data_class),
})
funnel.index.name = "data_class"
funnel


In [ ]:
steps = pd.DataFrame({
    "шаг": ["сырых → сошёлся", "сошёлся → каты", "каты → проскорено"],
    **{domain: [100 * funnel.loc[domain, "фит_сошёлся"] / funnel.loc[domain, "сырых"],
                100 * funnel.loc[domain, "каты_реконструкции"]
                / funnel.loc[domain, "фит_сошёлся"],
                100 * funnel.loc[domain, "проскорено"]
                / funnel.loc[domain, "каты_реконструкции"]]
       for domain in ORDER},
}).set_index("шаг").round(1)
steps


In [ ]:
stages = ["сырых", "фит_сошёлся", "каты_реконструкции", "проскорено"]
fig, axis = plt.subplots(figsize=(8, 4))
for k, (domain, colour) in enumerate(zip(ORDER,
                                         ("#2a78d6", "#eb6834", "#1baf7a", "#7b52ab"))):
    axis.bar([i + (k - 1.5) * 0.2 for i in range(len(stages))],
             [funnel.loc[domain, stage] for stage in stages],
             width=0.2, color=colour, label=domain)
axis.set_yscale("log")
axis.set_xticks(range(len(stages)))
axis.set_xticklabels(["сырые", "фит сошёлся", "каты реконструкции", "проскорено"])
axis.set_ylabel("событий (лог. шкала)")
axis.set_title("Воронка отборов по доменам")
axis.grid(alpha=0.15, axis="y")
axis.legend(frameon=False, fontsize=8)
fig.tight_layout()
plt.show()


In [ ]:
scored_mask = events.score.notna()
scored = pd.DataFrame({
    "data_class": events.data_class[scored_mask],
    "сошёлся": events.status[scored_mask] == "ok",
    "bin": np.minimum(49, np.floor(events.score[scored_mask] * 50)).astype(int),
})
hist = (scored.groupby(["data_class", "сошёлся", "bin"], observed=True).size()
        .rename("событий").reset_index())

EDGES = np.linspace(0, 1, 51)
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True, sharey=True)
for axis, domain in zip(axes.ravel(), ORDER):
    block = hist[hist.data_class == domain]
    for label, part, colour in (("все проскоренные", block, "#8c8c8c"),
                                ("из них фит сошёлся", block[block.сошёлся], "#2a78d6")):
        values = np.zeros(len(EDGES) - 1)
        np.add.at(values, part.bin.to_numpy(), part.событий.to_numpy())
        axis.stairs(values / values.sum(), EDGES, color=colour, lw=1.8,
                    label=f"{label} ({int(values.sum()):,})")
    axis.set_title(domain, fontsize=10)
    axis.set_yscale("log")
    axis.legend(fontsize=7.5, frameon=False, loc="best")
    axis.grid(alpha=0.15)
for axis in axes[:, 0]:
    axis.set_ylabel("доля событий группы (лог. шкала)")
for axis in axes[1, :]:
    axis.set_xlabel("скор классификатора")
fig.suptitle("Скор проскоренных событий: все и те, у кого фит сошёлся", fontsize=11)
fig.tight_layout()
plt.show()


In [ ]:
profile = pd.DataFrame({
    "data_class": events.data_class[scored_mask],
    "bin": np.minimum(24, np.floor(events.score[scored_mask] * 25)).astype(int),
    "сошлось": (events.status[scored_mask] == "ok").to_numpy(),
})
profile = (profile.groupby(["data_class", "bin"], observed=True)
           .agg(событий=("сошлось", "size"), сошлось=("сошлось", "sum"))
           .reset_index())
profile["доля"] = profile.сошлось / profile.событий
profile["ошибка"] = np.sqrt(profile.доля * (1 - profile.доля) / profile.событий)

fig, axis = plt.subplots(figsize=(7.5, 4.2))
for domain, colour in zip(ORDER, ("#2a78d6", "#eb6834", "#1baf7a", "#7b52ab")):
    block = profile[(profile.data_class == domain) & (profile.событий >= 200)]
    block = block.sort_values("bin")
    axis.errorbar((block.bin + 0.5) / 25, block["доля"], yerr=block.ошибка,
                  color=colour, lw=1.6, marker="o", ms=3, capsize=1.5,
                  elinewidth=0.6, label=domain)
axis.set(xlabel="скор классификатора", ylabel="доля событий с сошедшимся фитом",
         ylim=(0, 1))
axis.grid(alpha=0.15)
axis.set_title("Доля сошедшихся как функция скора (бины с ≥ 200 событиями)")
axis.legend(frameon=False, fontsize=8, ncol=4, loc="upper center",
            bbox_to_anchor=(0.5, -0.15))
fig.tight_layout()
plt.show()


In [ ]:
ok = events.status == "ok"
by_reko = ok & (events.n_hits >= 8) & (events.n_strings >= 3)
by_sn = by_reko & (events.n_sn_hits >= 8) & (events.n_sn_strings >= 3)

h8s3 = pd.DataFrame({
    "кат_по_реконструкции": by_class(covered & by_reko, events.data_class),
    "из_них_и_по_SN": by_class(covered & by_sn, events.data_class),
})
h8s3["согласие_процент"] = (100 * h8s3.из_них_и_по_SN
                            / h8s3.кат_по_реконструкции).round(1)
h8s3


In [ ]:
statuses = (pd.DataFrame({"data_class": events.data_class[covered],
                          "status": events.status[covered]})
            .groupby(["data_class", "status"], observed=True).size()
            .rename("событий").reset_index())
statuses["процент"] = (
    100 * statuses.событий
    / statuses.groupby("data_class", observed=True).событий.transform("sum")).round(1)
statuses.sort_values(["data_class", "событий"], ascending=[True, False]).reset_index(drop=True)


In [ ]:
events.data_class.value_counts().sort_index().to_frame("событий").assign(процент=lambda df: (100 * df.событий / df.событий.sum()).round(1))

In [ ]:
exp.n_sn_hits.isna().sum()#

In [ ]:
exp.head()

In [ ]:
def hitsstrings_mask(df: pd.DataFrame):
    mask = np.ones(len(df), dtype=bool)
    mask &= (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)
    return mask

def _apply_mask(df: pd.DataFrame):
    mask = np.ones(len(df), dtype=bool)
    
    mask &= (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)
    
    mask &= (df.score>=0.0)
    #mask &= (df.logit>=4.2)

    # mask &= (df.n_hits>=8)
    # mask &= (df.n_strings>=3)
    
    #mask &= (df.status == 'ok')
    #mask &= (df.logit >= 3.5)

    #mask &= (df.cov_matrix_status == 3)
    
    #mask &= (df.z_dist > 70)
    # mask &= (df.scf_max_theta < 1.7)
    # mask &= (df.scf_min_theta > 2.2)
    # mask &= (np.log10(df["time_chi2"] / (df.n_hits - 5)) < 1.0)

    # mask &= ((df["n_triplets"] / (df.n_hits - 5)) > 0.1)
    # mask &= ((df["q_acc_20"] / df["q_tot_20"]) > 0.05)
    # mask &= (df.t_res_30_med < 60)
    # mask &= (df["log_phit_30_min"] > -9)
    # mask &= (df.theta_err <= 2.5)

    #mask &= (df["ev_center_z"] < 220)
    #mask &= (df.theta_rec >= 90)
    #mask &= (df.theta_rec <= 150)

    return df[mask].copy()


col_to_plot = "logit"
#x_range = (0, 1)
x_range = (-5.0, 5.0)
_hist_kwargs = dict(density=True, bins=100, alpha=0.5, histtype='step', log=True, range=x_range)
plt.figure(figsize=(15, 6))
plt.hist(_apply_mask(muatm)[col_to_plot], label=f"muatm, total={len(_apply_mask(muatm))} / {sum(hitsstrings_mask(muatm))}={len(_apply_mask(muatm))/sum(hitsstrings_mask(muatm))*100 if sum(hitsstrings_mask(muatm)) > 0 else 0:.3f}%", **_hist_kwargs)
plt.hist(_apply_mask(exp)[col_to_plot], label=f"exp, total={len(_apply_mask(exp))} / {sum(hitsstrings_mask(exp))}={len(_apply_mask(exp))/sum(hitsstrings_mask(exp))*100 if sum(hitsstrings_mask(exp)) > 0 else 0:.3f}%", **_hist_kwargs)
plt.hist(_apply_mask(nue2)[col_to_plot], label=f"nue2, total={len(_apply_mask(nue2))} / {sum(hitsstrings_mask(nue2))}={len(_apply_mask(nue2))/sum(hitsstrings_mask(nue2))*100 if sum(hitsstrings_mask(nue2)) > 0 else 0:.3f}%", ls='--', **_hist_kwargs)
plt.hist(_apply_mask(nuatm)[col_to_plot], label=f"nuatm, total={len(_apply_mask(nuatm))} / {sum(hitsstrings_mask(nuatm))}={len(_apply_mask(nuatm))/sum(hitsstrings_mask(nuatm))*100 if sum(hitsstrings_mask(nue2)) > 0 else 0:.3f}%", ls='--', **_hist_kwargs)
plt.xlabel(col_to_plot)
plt.ylabel("Density")
plt.legend()
plt.grid(ls=":")
plt.xlim(*x_range)
plt.show()

In [ ]:
def hitsstrings_mask(df: pd.DataFrame):
    mask = np.ones(len(df), dtype=bool)
    mask &= (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)
    return mask

def _apply_mask(df: pd.DataFrame):
    mask = np.ones(len(df), dtype=bool)
    mask &= (df.n_sn_hits>=8)
    mask &= (df.n_sn_strings>=3)
    mask &= (df.score>=0.0)

    # mask &= (df.n_hits>=8)
    # mask &= (df.n_strings>=3)
    mask &= (df.status == 'ok')
    #mask &= (df.cov_matrix_status == 3)
    
    #mask &= (df.z_dist > 70)
    #mask &= (df.scf_max_theta < 1.7)
    #mask &= (np.log10(df["time_chi2"] / (df.n_hits - 5)) < 1.0)

    # mask &= ((df["n_triplets"] / (df.n_hits - 5)) > 0.1)
    # mask &= ((df["q_acc_20"] / df["q_tot_20"]) > 0.05)
    # mask &= (df.t_res_30_med < 60)
    # mask &= (df["log_phit_30_min"] > -9)
    #mask &= (df.theta_err <= 2.5)

    # mask &= (df["ev_center_z"] < 220)
    mask &= (df.theta_rec <= 170)
    #mask &= (df.theta_rec >= 70)

    return df[mask].copy()


col_to_plot = "logit"
x_range = (-5.0, 4.1)
_hist_kwargs = dict(density=True, bins=80, alpha=0.5, histtype='step', log=True, range=x_range)
plt.figure(figsize=(15, 6))
plt.hist(_apply_mask(muatm)[col_to_plot], label=f"muatm, total={len(_apply_mask(muatm))} / {sum(hitsstrings_mask(muatm))}={len(_apply_mask(muatm))/sum(hitsstrings_mask(muatm))*100 if sum(hitsstrings_mask(muatm)) > 0 else 0:.3f}%", **_hist_kwargs)
plt.hist(_apply_mask(exp)[col_to_plot], label=f"exp, total={len(_apply_mask(exp))} / {sum(hitsstrings_mask(exp))}={len(_apply_mask(exp))/sum(hitsstrings_mask(exp))*100 if sum(hitsstrings_mask(exp)) > 0 else 0:.3f}%", **_hist_kwargs)
plt.hist(_apply_mask(nue2)[col_to_plot], label=f"nue2, total={len(_apply_mask(nue2))} / {sum(hitsstrings_mask(nue2))}={len(_apply_mask(nue2))/sum(hitsstrings_mask(nue2))*100 if sum(hitsstrings_mask(nue2)) > 0 else 0:.3f}%", ls='--', **_hist_kwargs)
plt.xlabel(col_to_plot)
plt.ylabel("Density")
plt.legend()
plt.grid(ls=":")
plt.xlim(*x_range)
plt.show()

In [ ]:
from excess import display as events_drawer
from inference.track_fit_alike_bars import fit_events, masks
import h5py


#: Что лежит в `raw/labels` -- смысл у источников РАЗНЫЙ, и это ловушка:
#:   mc_merged, mc_reco -- правда: код происхождения импульса (fMagic).
#:       0 -- шум, отрицательные -- свет мюонного трека, k*10**6 + j -- свет ливня k;
#:   exp_reco -- маска scanfit (0/1), то есть реконструкция BARS, а не правда;
#:   exp_full -- нули: у данных правды нет.
#: Поэтому `truth=True` для данных не «нарисует что-нибудь похожее», а упадёт.
LABELS_MEANING = {"mc_merged": "правда", "mc_reco": "правда",
                  "exp_reco": "маска scanfit", "exp_full": "ничего (нули)",
                  "exp": "ничего (нули)"}


def attach_hits(events, threshold=reader.SN_THRESHOLD, fits=None, truth=False):
    """Хиты выбранных событий -- по колонке-списку на величину.

    Рассчитано на горсть событий: читает срез `raw/data[start:end]` на каждое, а не
    пачками, как `reader.stream`. Для семи строк это правильный размен -- ридер
    оптимизирован под тысячи событий и сливает соседние в общие пролёты.

    Ждёт колонки `source`, `part_key`, `local_idx` и, для MC, `data_class` -- ровно то,
    что кладёт в кадр читалка. Индекс входного кадра сохраняется.

    `is_sig` берётся из того же файла вероятностей и тем же строгим `>` во float32,
    что и везде в пайплайне: с `>=` одно событие из двадцати тысяч получило бы другое
    число сигнальных хитов, чем записано в базе предсказаний.

    `fits` -- кадр от `fit_events(..., keep_objects=True)` для этих же событий, с тем
    же индексом; можно не передавать, если колонка `fit` с объектами `TrackFit` уже
    лежит в кадре. Тогда добавляется `is_fit` -- импульсы, взятые в трек реконструкцией.
    Порядок импульсов совпадает с этим чтением по построению: и `read_event_hits`,
    откуда фит берёт хиты, и этот срез режут один и тот же `raw/data[start:stop]`,
    поэтому `fit_hits` индексируют ровно эти массивы. Несшиеся события дают нули.

    `truth=True` -- только для МК-источников (`mc_merged`, `mc_reco`) -- добавляет три
    колонки по `raw/labels`: `is_truth` (любой свет, то есть не шум), `is_truth_track`
    (свет мюонного трека) и `is_truth_shower` (свет ливней). Для данных правды не
    существует, и попытка её попросить -- исключение, а не молчаливая подмена: в
    `exp_reco` в том же поле лежит маска scanfit, а в `exp_full` одни нули.
    """
    if fits is None and "fit" in getattr(events, "columns", ()):
        fits = events                      # объекты фита уже в кадре
    wanted = ["q", "t", "x", "y", "z", "channel", "prob", "is_sig"]
    if fits is not None:
        wanted.append("is_fit")
    if truth:
        wanted += ["is_truth", "is_truth_track", "is_truth_shower"]
    collected = {name: [None] * len(events) for name in wanted}
    positions = {key: index for index, key in enumerate(events.index)}

    grouping = ["source", "part_key"] + (["data_class"] if "data_class" in events else [])
    for keys, block in events.groupby(grouping, sort=False):
        keys = keys if isinstance(keys, tuple) else (keys,)
        source, part_key = keys[0], keys[1]
        if truth and LABELS_MEANING.get(source) != "правда":
            raise ValueError(
                f"{source}: в raw/labels не правда, а "
                f"{LABELS_MEANING.get(source, 'неизвестно что')}. Правда есть только "
                f"у mc_merged и mc_reco.")
        group_name = reader.spec.h5_group(source, keys[2]) if len(keys) > 2 else source
        specification = reader.spec.SOURCES[source]

        with h5py.File(specification.h5, "r") as store, \
             h5py.File(specification.probs, "r") as probabilities:
            hits = store[group_name]["raw"]
            boundaries = hits["ev_starts"][part_key]["data"]
            data = hits["data"][part_key]["data"]
            channels = hits["channels"][part_key]["data"]
            labels = hits["labels"][part_key]["data"] if truth else None
            probs = probabilities[group_name]["probs"][part_key]["data"]

            for label, local in zip(block.index, block.local_idx.to_numpy()):
                start, stop = int(boundaries[local]), int(boundaries[local + 1])
                block_of_hits = data[start:stop]
                probability = probs[start:stop]
                slot = positions[label]
                for column, name in enumerate(reader.HIT_VARS):
                    collected[name][slot] = block_of_hits[:, column].tolist()
                collected["channel"][slot] = channels[start:stop].tolist()
                collected["prob"][slot] = probability.tolist()
                collected["is_sig"][slot] = reader.signal_mask(probability, threshold).tolist()

                if truth:
                    code = labels[start:stop]
                    collected["is_truth"][slot] = (code != 0).tolist()
                    collected["is_truth_track"][slot] = (code < 0).tolist()
                    collected["is_truth_shower"][slot] = (code > 0).tolist()

                if fits is not None:
                    in_fit = np.zeros(stop - start, dtype=bool)
                    taken = getattr(fits["fit"].loc[label], "fit_hits", None)
                    if taken is not None and len(taken):
                        index = np.asarray(taken, dtype=np.int64)
                        if index.max() >= in_fit.size:
                            raise ValueError(
                                f"{label}: индекс хита {index.max()} вне события "
                                f"({in_fit.size} импульсов) -- кадр и фит разошлись")
                        in_fit[index] = True
                    collected["is_fit"][slot] = in_fit.tolist()

    return events.assign(**collected)

N = 3
take = _apply_mask(muatm)
take = take[take['logit']>=0]
take = take.sample(N, random_state=0)
take = attach_hits(
    take,
    fits=fit_events(take, n_workers=8, keep_objects=True),
    truth=False
    )
take = take.nlargest(N, "score")     # или .sample(3, random_state=0)
events = events_drawer.load(take)                  # колонки source/data_class/part_key/local_idx
fig = events_drawer.figure(events)                 # строка = событие
events_drawer.save(fig, dpi=300, name=f"muatm_strong_mask_strange{N}_2_weights")              # png + сайдкар, фигуру закрывает

In [ ]:
nue2.head()